# Gazette Package Driver

This is the recommended notebook entry point for the package workflow. It stays thin: the notebook selects inputs and output paths, while `gazette_mistral_pipeline` handles OCR replay, parsing, schema validation, and bundle writing.

The default path is offline replay. It does not require an API key, network access, `.env`, live Mistral calls, or historical `prototype_outputs`.


In [13]:
from pathlib import Path
import sys

cwd = Path.cwd()
repo_root = next(
    (
        candidate
        for candidate in (cwd, cwd.parent)
        if (candidate / "gazette_mistral_pipeline" / "__init__.py").is_file()
    ),
    None,
)
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from gazette_mistral_pipeline import (
        Bundles,
        GazetteConfig,
        get_envelope_schema,
        parse_file,
        parse_url,
        validate_envelope_json,
        write_envelope,
    )
except ModuleNotFoundError as exc:
    if exc.name == "pydantic" and repo_root is not None:
        raise ModuleNotFoundError(
            "This notebook kernel is missing package dependencies. "
            "Run this in a notebook cell, then restart the kernel: "
            f'%pip install -e "{repo_root}"'
        ) from exc
    raise


ModuleNotFoundError: This notebook kernel is missing package dependencies. Run this in a notebook cell, then restart the kernel: %pip install -e "c:\Users\Ronald Wahome\Documents\gazette_mistral_pipeline"

## Offline Replay Configuration

Run this notebook from the repository root or from the `examples/` directory. The checked-in replay fixture is intentionally tiny and non-secret; it exercises the package API without contacting Mistral.


In [ ]:
PDF_URL = "https://new.kenyalaw.org/akn/ke/officialGazette/2026-04-17/68/eng@2026-04-17/source.pdf"

cwd = Path.cwd()
examples_dir = cwd / "examples"
if not (examples_dir / "tiny_replay.raw.json").is_file() and (cwd / "tiny_replay.raw.json").is_file():
    examples_dir = cwd

replay_path = examples_dir / "tiny_replay.raw.json"
output_root = examples_dir / "_example_outputs"
stage_dir = output_root / "stage"
bundle_dir = output_root / "bundles"

config = GazetteConfig(
    runtime={
        "replay_raw_json_path": replay_path,
        "output_dir": stage_dir,
    }
)


## Parse, Validate, And Write Bundles

`parse_url` reads the replay fixture because `runtime.replay_raw_json_path` is configured. `write_envelope` is the only call that writes final bundles.


In [ ]:
env = parse_url(PDF_URL, config=config)

written = write_envelope(
    env,
    bundle_dir,
    Bundles(notices=True, tables=True, document_index=True, schema=True),
)

validated = validate_envelope_json(written["envelope"])
schema = get_envelope_schema()

{
    "run_name": validated.source.run_name,
    "notice_count": validated.stats.notice_count,
    "written_bundles": sorted(written),
    "schema_title": schema.get("title"),
}


## Optional Live Mistral URL OCR

Live OCR is intentionally not executed by default. To run it manually, set `MISTRAL_API_KEY` in the process environment, choose an explicit output directory, and opt in with `allow_live_mistral=True`. Live local PDF upload is not implemented in this package yet; local PDFs should use replay mode until a later approved feature adds upload support.


In [ ]:
# live_config = GazetteConfig(
#     runtime={
#         "allow_live_mistral": True,
#         "output_dir": examples_dir / "_live_outputs" / "stage",
#     }
# )
# live_env = parse_url(PDF_URL, config=live_config)
# live_written = write_envelope(live_env, examples_dir / "_live_outputs" / "bundles")


## Local PDF Replay Shape

`parse_file` is imported for local PDF replay workflows. Supply a local PDF path and keep `runtime.replay_raw_json_path` configured; do not use local PDF live OCR until upload support is added.


In [ ]:
# local_pdf = Path("/path/to/local-gazette.pdf")
# local_env = parse_file(local_pdf, config=config)
